# T03 · Calcul différentiel & optimisation — ADIF72

**Objectifs :**
- Dérivées, règle de la chaîne, gradients
- Descente de gradient (à la main, puis ML)
- Convexité (Hessian, valeurs propres)
- Régression linéaire et logistique par descente de gradient

**Pourquoi c'est le cœur du ML :**
Tous les modèles d'apprentissage profond et les SVM, réseaux de neurones,
ensembles (forets), etc. s'entraînent par **descente de gradient** (ou ses
variantes : momentum, Adam, L-BFGS…). Comprendre le calcul du gradient est
indispensable.

In [ ]:
import numpy as np
from scipy import optimize
np.set_printoptions(precision=4, suppress=True, linewidth=100)
print("Prêt. numpy", np.__version__, " scipy", optimize.__name__[:10])

## 1 · Dérivées & gradient (analytique et numérique)
### Dérivée analytique (règles)
```
f(x) = x³   ->  f'(x) = 3x²
(f·g)' = f'g + fg'
(f∘g)'(x) = f'(g(x)) · g'(x)    (règle de la chaîne)
```

### Dérivée numérique (différence centrale)
```
f'(x) ≈ (f(x+h) - f(x-h)) / (2h)
```
Précision **O(h²)** — bien meilleure que la différence avant `(f(x+h)-f(x))/h`
(précision O(h)). On prend habituellement `h ≈ 1e-5`.

**Pourquoi ça compte :** en deep learning, on n'a pas toujours la forme
analytique du gradient ; on peut vérifier par différences (c'est le
« *gradient check* »).

In [ ]:
def f(x):
    return x ** 3
def df(x):
    return 3 * x ** 2

def derive_numerique(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

for xv in [0, 1, 2, 3]:
    print(f"x={xv}:  f' analytique={df(xv):.4f}   numérique={derive_numerique(f, xv):.4f}")
assert np.allclose([df(xv) for xv in [0, 1, 2]],
                   [derive_numerique(f, xv) for xv in [0, 1, 2]])

# Règle de la chaîne : g(x) = sin(x^2)  ->  g'(x) = cos(x^2)*2x
x = 0.7
g_prime = np.cos(x ** 2) * 2 * x
print(f"\nChaîne : g'(0.7) = {g_prime:.4f}")
print(f"Vérif numé  : {derive_numerique(lambda t: np.sin(t**2), x):.4f}")
assert np.isclose(g_prime, derive_numerique(lambda t: np.sin(t ** 2), x))

### Gradient

Pour `f : ℝⁿ → ℝ`, le **gradient** est le vecteur des dérivées partielles :
```
∇f = (∂f/∂x₁, ∂f/∂x₂, ..., ∂f/∂xₙ)ᵀ
```
Il pointe dans la direction de la **plus forte croissance** de `f`.
Pour **minimiser**, on va dans la direction opposée : `-∇f`.

In [ ]:
# f(x,y) = x² + 3xy + y²
# df/dx = 2x + 3y ;  df/dy = 3x + 2y
def grad_f(x, y):
    return np.array([2 * x + 3 * y, 3 * x + 2 * y])

g = grad_f(1.0, 2.0)
print("∇f(1,2) =", g)

# Vérif par différences
h = 1e-5
num = np.array([
    ((1+h)**2 + 3*(1+h)*2 + 4 - ((1-h)**2 + 3*(1-h)*2 + 4)) / (2*h),
    (1 + 3*1*(2+h) + (2+h)**2 - (1 + 3*1*(2-h) + (2-h)**2)) / (2*h)])
print("numé      =", num)
assert np.allclose(g, num, atol=1e-3)

### Exercice corrigé

In [ ]:
# ---- Exercice P1 : f'(x) de exp(-x^2), analytique vs numerique ----
x0 = 0.5
f1_ = lambda x: np.exp(-x**2)
df1_ = lambda x: -2 * x * np.exp(-x**2)
h = 1e-6
num = (f1_(x0+h) - f1_(x0-h)) / (2*h)
print(f"P1-ex : f'(0.5) analytique = {df1_(x0):.6f}, numerique = {num:.6f}")
assert np.isclose(df1_(x0), num, atol=1e-6)


## 2 · Descente de gradient, convexité & Hessian
Algorithme :
```
x_{t+1} = x_t - α ∇f(x_t)
```
- `α` : **pas de learning rate**. Trop petit = convergence lente.
  Trop grand = divergence / oscillation.
- On choisit souvent `α` par « *trial and error* » ou par **rule of thumb** :
  `α < 2 / L` où `L` est le constant de Lipschitz du gradient.

Exemple 1D : `f(x) = (x-3)² + (x+1)² = 2x² - 4x + 10`. Minimum à `x* = 1`,
`f(x*) = 8`.

In [ ]:
def f1(x):
    return (x - 3) ** 2 + (x + 1) ** 2
def df1(x):
    return 4 * x - 4   # dérivée analytique

def gradient_descent_1d(f, df, x0, lr=0.1, n_iter=200, tol=1e-8):
    x = float(x0)
    hist = [x]
    for i in range(n_iter):
        g = df(x)
        x = x - lr * g
        hist.append(x)
        if abs(g) < tol:
            print(f"  convergé en {i+1} itérations")
            break
    return np.array(hist)

hist = gradient_descent_1d(f1, df1, x0=-5.0, lr=0.1)
print(f"x0 = -5.0, f(x0) = {f1(-5):.2f}")
print(f"x* ≈ {hist[-1]:.4f}, f(x*) ≈ {f1(hist[-1]):.4f}   (optimum : 1, 8)")
assert np.isclose(hist[-1], 1.0, atol=1e-3)

### Convexité & Hessian

### Test de convexité (matrice symétrique)
Une fonction différentiable à deux fois est **convexe** si son
**Hessian** `H` (matrice des 2ndes dérivées) est semi-défini positif
(les valeurs propres de `H` sont `≥ 0`).
Si les valeurs propres sont **strictement** `> 0`, on est en
**strictement convexe** : le minimum est **unique**.

### Pourquoi ça compte
- Descente de gradient converge globalement pour des fonctions convexes
- Les problèmes de régression linéaire (moindres carrés) sont convexes
- Les réseaux de neurones **ne le sont pas** — d'où les minima locaux
  et le plateau de gradient

In [ ]:
# g(x,y) = x² + y² - 2x - 4y + 10  (convexe : Hessian = 2I > 0)
H = np.array([[2, 0], [0, 2]])
print("Hessian :\n", H, "  valeurs propres =", np.linalg.eigvalsh(H))

res = optimize.minimize(lambda v: v[0]**2 + v[1]**2 - 2*v[0] - 4*v[1] + 10,
                        x0=[0., 0.], jac=lambda v: [2*v[0]-2, 2*v[1]-4])
print("Minimum scipy :", res.x, "  valeur :", res.fun)
assert np.allclose(res.x, [1, 2]) and np.isclose(res.fun, 5)

### Exercice corrigé

In [ ]:
# ---- Exercice P2 : points critiques de g(x) = x^4 - 2x^2 ----
g2 = lambda x: x**4 - 2*x**2
crits = np.roots([4, 0, -4, 0])
print(f"P2-ex : points critiques = {crits}")
for c in crits:
    print(f"    g({c:.2f}) = {g2(c):.3f},  g''({c:.2f}) = {4*c**2-4:.3f}")
assert len(crits) == 3 and np.isclose(min(g2(c) for c in crits), -1.0)


## 3 · Régression linéaire par descente de gradient
Modèle : `ŷ = θ x + b`. Perda : MSE `L = (1/n) Σ (ŷᵢ - yᵢ)²`.
Gradients :
```
∂L/∂θ = (2/n) Σ xᵢ (θxᵢ + b - yᵢ)
∂L/∂b = (2/n) Σ (θxᵢ + b - yᵢ)
```
C'est la même chose que les moindres carrés, mais **itératif** (indispensable
si les données ne tiennent pas en mémoire, ou si la matrice est énorme).

In [ ]:
rng = np.random.default_rng(42)
n = 200
X = rng.uniform(-5, 5, n)
theta_true, b_true = 2.5, 1.0
y = theta_true * X + b_true + rng.normal(0, 0.5, n)

def loss_mse(theta, b, X, y):
    pred = theta * X + b
    return np.mean((pred - y) ** 2)

def grad_mse(theta, b, X, y):
    pred = theta * X + b
    err = pred - y
    return (2/len(y)) * np.sum(X * err), (2/len(y)) * np.sum(err)

theta, b = 0.0, 0.0
lr = 0.01
for it in range(2000):
    gt, gb = grad_mse(theta, b, X, y)
    theta -= lr * gt
    b -= lr * gb
print(f"θ estimé = {theta:.3f} (vrai={theta_true}),  b estimé = {b:.3f} (vrai={b_true})")
print(f"Loss finale = {loss_mse(theta, b, X, y):.4f}")
assert abs(theta - theta_true) < 0.1 and abs(b - b_true) < 0.1

### Exercice corrigé

In [ ]:
# ---- Exercice P3 : pente/ordonnee par moindres carres a la main ----
xr = np.array([0.0, 1, 2, 3, 4]); yr = np.array([1.0, 3, 4, 5, 7])
beta_ = ((xr-xr.mean())*(yr-yr.mean())).sum() / ((xr-xr.mean())**2).sum()
alpha_ = yr.mean() - beta_ * xr.mean()
print(f"P3-ex : beta = {beta_:.2f}, alpha = {alpha_:.2f}")
assert np.isclose(beta_, 1.4) and np.isclose(alpha_, 1.2)


## 4 · Logistique, Jacobien & backpropagation *(MML ch. 5.3-5.6)*
Modèle : `ŷ = σ(xᵀw + b)` avec `σ(z) = 1/(1 + e^{-z})`.
Perda (cross-entropy) :
```
L = -(1/n) Σ [ yᵢ log(ŷᵢ) + (1-yᵢ) log(1-ŷᵢ) ]
```
Gradients :
```
∂L/∂w = (1/n) Xᵀ (ŷ - y)     (le même résidu que la régression linéaire !)
∂L/∂b = (1/n) Σ (ŷ - y)
```
C'est ce que fait sklearn.linear_model.LogisticRegression. Implémenter à la main
→ comprendre comment le modèle apprend le **frontière de décision**.

In [ ]:
X2 = rng.normal(0, 1, (n, 2))
w_true = np.array([1.5, -2.0])
p = 1 / (1 + np.exp(-(X2 @ w_true + 0.5)))
y2 = (rng.uniform(size=n) < p).astype(int)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def log_loss(w, b, X, y):
    z = sigmoid(X @ w + b)
    eps = 1e-12
    return -np.mean(y * np.log(z + eps) + (1 - y) * np.log(1 - z + eps))

def log_loss_grad(w, b, X, y):
    z = sigmoid(X @ w + b)
    err = z - y
    return (1/len(y)) * (X.T @ err), (1/len(y)) * np.sum(err)

w, b = np.zeros(2), 0.0
for it in range(2000):
    gw, gb = log_loss_grad(w, b, X2, y2)
    w -= 0.1 * gw
    b -= 0.1 * gb
acc = np.mean((sigmoid(X2 @ w + b) >= 0.5).astype(int) == y2)
print(f"Vrais poids = {w_true},  estimés = {w}")
print(f"Accuracy = {acc:.3f}   (perdita = {log_loss(w, b, X2, y2):.4f})")
assert acc > 0.7

# Comparaison avec sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
lr_model = LogisticRegression(max_iter=1000).fit(X2, y2)
print(f"sklearn : weights={lr_model.coef_[0]}, accuracy="
      f"{accuracy_score(y2, lr_model.predict(X2)):.3f}")

### Jacobien & gradients de fonctions vectorielles *(MML ch. 5.3-5.4)*

Pour `f : ℝⁿ → ℝᵐ`, le **Jacobian** est la matrice `m×n` des dérivées
partielles : `Jᵢⱼ = ∂fᵢ/∂xⱼ`. C'est le **linéarisation de premier ordre**
de `f` (analogue du gradient pour les fonctions scalaires).

Règles essentielles :
- **chaîne** : `J_{g∘f} = J_g · J_f` (le backbone de la backpropagation)
- le gradient d'une fonction scalaire `ℒ(x)` **est** le Jacobien
  transposé vu comme un vecteur : `∇ℒ = J_ℒᵀ`
- en ML on ne calcule presque jamais `J` entier : on calcule des
  **produits vecteur-Jacobien (VJP)** — c'est exactement ce que fait
  la rétropropagation.

*À l'examen* : écrire `J` à la main pour une 2-3 variables, vérifier
par différences finies.

In [ ]:
def f(xy):
    x, y = xy
    return np.array([x**2 * y, np.sin(x * y), x + y**3])

def J_f(x, y):
    return np.array([[2*x*y,        x**2],
                     [y*np.cos(x*y), x*np.cos(x*y)],
                     [1,            3*y**2]])

pt = np.array([0.7, -0.4])
J = J_f(*pt)
h = 1e-6
J_num = np.zeros((3, 2))
for j in range(2):
    dp = np.zeros(2); dm = np.zeros(2); dp[j] = h; dm[j] = -h
    J_num[:, j] = (f(pt + dp) - f(pt + dm)) / (2 * h)
print("J (analytique) =\n", J)
print("J (numérique)  =\n", J_num)
assert np.allclose(J, J_num, atol=1e-5)

# Chaîne : f(x) = ||g(x)||², g(x,y) = (sin(x), x·y)
# d(f∘g)/dx = 2 g· g'  ->  J_{f∘g} = J_f(g) @ J_g
x, y = 0.6, 0.3
def gfun(xy):
    x, y = xy
    return np.array([np.sin(x), x * y])
Jg = np.array([[np.cos(x), 0.0], [y, x]])   # Jacobien de g
gval = gfun((x, y))
# h(x,y) = ||g(x,y)||²  ;  dh/dx = 2 * g · (dg/dx)
dhdx = 2.0 * (gval[0] * Jg[0, 0] + gval[1] * Jg[1, 0])
hfun = lambda xy: np.linalg.norm(gfun(xy)) ** 2
dhdx_num = (hfun((x + h, y)) - hfun((x - h, y))) / (2 * h)
print(f"\nChaîne : dh/dx (analytique) = {dhdx:.5f}")
print(f"          (numérique)        = {dhdx_num:.5f}")
assert np.isclose(dhdx, dhdx_num, atol=1e-5)

### Backpropagation & dérivées automatiques *(MML ch. 5.6)*

La **rétropropagation** n'est qu'une application *efficace* de la règle
de la chaîne sur un graphe de calcul : on propage les dérivées de la
perte vers les paramètres, en réutilisant les valeurs du *forward pass*.

Complexité : **O(définition du réseau)** — contre O(poids) si on
recalcule chaque dérivée par différences finies.

**Gradient check** (bonne pratique d'implémentation) : comparer le
gradient analytique (backprop) au gradient numérique (différence
centrale). S'ils ne sont pas égaux à `1e-6` près → bug dans la
backprop.

In [ ]:
rng = np.random.default_rng(0)
x_in = np.array([0.5, -0.3])
y_true = 1.0
W1 = np.array([[0.5, 0.1, -0.4],
                  [-0.2, 0.8, 0.6]])                # 2 -> 3
b1 = np.array([0.1, -0.2, 0.0])
W2 = np.array([0.3, -0.7, 0.9])                        # 3 -> 1
b2 = 0.2

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def net_loss(W1, b1, W2, b2):
    z1 = x_in @ W1 + b1
    a1 = np.tanh(z1)
    z2 = a1 @ W2 + b2
    a2 = sigmoid(z2)
    return (a2 - y_true) ** 2, a1, a2

loss, a1, a2 = net_loss(W1, b1, W2, b2)
# ---- backprop (chaîne de la règle) ----
delta_out = 2 * (a2 - y_true) * a2 * (1 - a2)      # dL/da2
dW2 = delta_out * a1                               # da2/dW2 = a1
db2 = delta_out
dz1 = (W2 * delta_out) * (1 - a1 ** 2)             # da1/dz1 = 1 - tanh²
dW1 = np.outer(x_in, dz1)
db1 = dz1
# ---- gradient check numérique sur W1 ----
h = 1e-6
dW1_num = np.zeros_like(W1)
for i in range(2):
    for j in range(3):
        dp = W1.copy(); dm = W1.copy()
        dp[i, j] += h; dm[i, j] -= h
        dW1_num[i, j] = (net_loss(dp, b1, W2, b2)[0]
                         - net_loss(dm, b1, W2, b2)[0]) / (2 * h)
print("dL/dW1 analytique :\n", dW1)
print("dL/dW1 numérique  :\n", dW1_num)
assert np.allclose(dW1, dW1_num, atol=1e-6)
print("\nGradient check OK : backprop = règle de la chaîne.")

### Exercice corrigé

In [ ]:
# ---- Exercice P4 : gradient de f(x,y) = x^2 * y en (2,3) + gradient check ----
f_ = lambda x, y: x**2 * y
dfx, dfy = 2*2*3, 2**2
h = 1e-6
numx = (f_(2+h, 3) - f_(2-h, 3)) / (2*h)
numy = (f_(2, 3+h) - f_(2, 3-h)) / (2*h)
print(f"P4-ex : grad analytique = ({dfx}, {dfy}), numerique = ({numx:.4f}, {numy:.4f})")
assert np.isclose(dfx, numx, atol=1e-4) and np.isclose(dfy, numy, atol=1e-4)


## 5 · Hessian, Taylor & Newton *(MML ch. 5.7-5.8)*
**Linéarisation de Taylor d'ordre 2** autour de `x₀` :
```
f(x₀ + v) ≈ f(x₀) + ∇f(x₀)ᵀ v + ½ vᵀ H(x₀) v
```
Outils :
- **Hessian** `H` : matrice des 2ndes dérivées ; signature des valeurs
  propres → minimum local (SP), maximum (SN), point selle (indéfinie)
- **pas de Newton** : `x ← x - H⁻¹ ∇f` (convergence quadratique près
  du minimum, mais coûteux)
- **Hessian-vector (HVP)** : `H @ v` sans construire H — utilisé par
  L-BFGS, natural gradient, et la double backpropagation

In [ ]:
# f(x) = ½ xᵀAx - bᵀx (quadratique) : Taylor ordre 2 EXACT
Aq = np.array([[3., 1.], [1., 2.]]); bq = np.array([1., -1.])
fq = lambda v: 0.5 * v @ Aq @ v - bq @ v
x0 = np.array([0.3, -0.2]); v = np.array([0.5, 0.4])
grad = Aq @ x0 - bq
H = Aq
f_true = fq(x0 + v)
T1 = fq(x0) + grad @ v
T2 = fq(x0) + grad @ v + 0.5 * v @ H @ v
print(f"f(x0+v)        = {f_true:.5f}")
print(f"Taylor ordre 1 = {T1:.5f}   (erreur {abs(f_true-T1):.4f})")
print(f"Taylor ordre 2 = {T2:.5f}   (erreur {abs(f_true-T2):.2e} -> exact pour quadratique)")
assert np.isclose(T2, f_true)

# Pas de Newton : convergence en 1 pas pour une quadratique
x_new = x0 - np.linalg.solve(H, grad)
print(f"\nPas Newton : x* = {x_new} (minimum exact de f quadratique)")
assert np.allclose(x_new, np.linalg.solve(Aq, bq))

# Hessian d'une fonction NON quadratique : test de signature
# f(x,y) = x⁴/4 - x² + xy + y⁴/4  ->  H = [[3x²-2, 1],[1, 3y²]]
def H_analytique(x, y):
    return np.array([[3*x**2 - 2, 1.0], [1.0, 3*y**2]])
Hpt = H_analytique(1.0, -0.5)
print("\nH en (1, -0.5) =\n", Hpt, " valeurs propres =", np.linalg.eigvalsh(Hpt))
print("H indéfini (une +, une -) -> POINT SELLE")
assert np.linalg.eigvalsh(Hpt)[0] < 0 < np.linalg.eigvalsh(Hpt)[1]

# HVP par différences du gradient (double backprop en pratique)
def f4(x, y):
    return x**4/4 - x**2 + x*y + y**4/4
def g4(x, y):  # gradient de f = x^4/4 - x^2 + x*y + y^4/4
    return np.array([x**3 - 2*x + y, x + y**3])
hp = 1e-4
vp = np.array([0.7, -0.3])
Hvp = H_analytique(1.0, -0.5) @ vp
Hvp_num2 = (g4(1.0 + hp*vp[0], -0.5 + hp*vp[1]) - g4(1.0 - hp*vp[0], -0.5 - hp*vp[1])) / (2*hp)
print(f"H @ v (analytique) = {Hvp},  HVP numérique = {Hvp_num2}")
assert np.allclose(Hvp, Hvp_num2, atol=1e-4)

### Exercice corrigé

In [ ]:
# ---- Exercice P5 : Newton sur x^2 - 4 (x0 = 3) ----
x_ = 3.0
for k in range(2):
    x_ = (x_ + 4/x_) / 2
    print(f"P5-ex : x{k+1} = {x_:.6f}")
assert np.isclose(x_, 313/156, atol=1e-4)
print("      x1 = 13/6, x2 = 313/156 -> convergence quadratique vers 2")


## 6 · Optimisation contrainte — Lagrange/KKT *(MML ch. 7.2)*
Problème : minimiser `f(x)` sous contrainte `g(x) = 0`.
Idea : au point optimal, **le gradient de f est colinéaire au gradient
de g** (sinon on pourrait descendre f en se déplaçant le long de la
variété de contrainte).

**Théorème de Lagrange** : il existe `λ` tel que
```
∇f(x*) + λ ∇g(x*) = 0      (conditions d'optimalité, KKT dans le cas général)
```
On résout le système d'équations `∇f + λ∇g = 0` + `g(x*) = 0`
(`n+1` équations, `n+1` inconnues).

Pour les **inégalités** `g(x) ≤ 0`, on ajoute les conditions de **complémentarité** :
`λ ≥ 0`, `λ g(x*) = 0` (la contrainte est soit inactive λ=0, soit active
`g(x*)=0`). C'est la base des SVM (T07 / MML ch. 12) et de la régularisation.

In [ ]:
# Exemple 1 : min x² + y²  s.t.  x + y = 2
# ∇f = λ∇g : (2x, 2y) = λ(1, 1) -> x = y ;  x + y = 2 -> x* = y* = 1, f* = 2, λ* = 2
from scipy.optimize import minimize
res = minimize(lambda vv: vv[0]**2 + vv[1]**2, x0=[5., 0.],
               constraints={'type': 'eq', 'fun': lambda vv: vv[0] + vv[1] - 2},
               method='SLSQP')
print("Lagrange : x*=1, y*=1, f*=2 (λ*=2)")
print("scipy SLSQP :", res.x, " valeur :", res.fun)
assert np.allclose(res.x, [1, 1]) and np.isclose(res.fun, 2)

# Exemple 2 (inégalité, complémentarité) :
# min (x-3)² + (y-4)²  s.t.  x² + y² ≤ 1
# minimum sans contrainte (3,4) est hors du disque -> optimum SUR le bord
# point du cercle unitaire le plus proche de (3,4) : (3,4)/||(3,4)|| = (0.6, 0.8)
# f* = ||(3,4) - (0.6,0.8)||² = 2.4² + 3.2² = 16
res2 = minimize(lambda vv: (vv[0]-3)**2 + (vv[1]-4)**2, x0=[0.9, 0.1],
                constraints=[{'type': 'ineq', 'fun': lambda vv: 1 - vv[0]**2 - vv[1]**2}],
                method='SLSQP')
print("\nInégalité :", res2.x, " valeur :", res2.fun)
assert np.allclose(res2.x, [0.6, 0.8]) and np.isclose(res2.fun, 16.0)
print("f* = 16 : le gradient de f en (0.6,0.8) est colinaire au gradient de g")
gf = 2 * np.array([0.6 - 3, 0.8 - 4])
gg = 2 * np.array([0.6, 0.8])
assert np.allclose(gf, (gf @ gg / (gg @ gg)) * gg, atol=1e-6)  # colinéarité

### Exercice corrigé

In [ ]:
# ---- Exercice P6 : Lagrange min x^2 + y^2 s.t. x + y = 4 ----
xstar, ystar = 2.0, 2.0
print(f"P6-ex : x* = y* = {xstar}, f* = {xstar**2 + ystar**2}, lambda* = {2*xstar}")
assert xstar**2 + ystar**2 == 8


## 7 · Récapitulatif — choisir l'optimiseur
| Fonction | Caractéristique | Méthode |
|---|---|---|
| Polynôme de bas degré | convexe, gradient lisse | GD, Newton |
| Réseau de neurones | non-convexe, gradient stochastique | Adam, SGD+momentum |
| SVM | convexe, gros hessian | SMO, L-BFGS |
| Linéaire (moindres carrés) | convexes, gradient exact | GD, closed form |
| Logistique | convexes | Newton / IRLS / GD |


### Exercice corrigé

In [ ]:
# ---- Exercice P7 : Hessian de h(x,y) = x^2 - 2y^2 (selle) ----
H_ = np.array([[2.0, 0], [0, -2]])
ev = np.linalg.eigvalsh(H_)
print(f"P7-ex : valeurs propres = {ev} -> indefini ? {bool(ev[0]<0<ev[1])} (pas de minimum)")
assert ev[0] < 0 < ev[1]
print("\n✅ T03 termine — 7 points, exercices integres.")
